In [1]:
import pandas as pd

# Load the March 2024 file
df = pd.read_csv("March-2024-revised-130624-abcdef123.csv")

# How big is it? (rows, columns)
print("Rows and columns:", df.shape)

# Show the first 5 rows
df.head()

Rows and columns: (198, 22)


,Period,Org Code,Parent Org,Org name,A&E attendances Type 1,A&E attendances Type 2,A&E attendances Other A&E Department,A&E attendances Booked Appointments Type 1,A&E attendances Booked Appointments Type 2,A&E attendances Booked Appointments Other Department,...,Attendances over 4hrs Other Department,Attendances over 4hrs Booked Appointments Type 1,Attendances over 4hrs Booked Appointments Type 2,Attendances over 4hrs Booked Appointments Other Department,Patients who have waited 4-12 hs from DTA to admission,Patients who have waited 12+ hrs from DTA to admission,Emergency admissions via A&E - Type 1,Emergency admissions via A&E - Type 2,Emergency admissions via A&E - Other A&E department,Other emergency admissions
0,MSitAE-MARCH-2024,NQTE4,NHS ENGLAND MIDLANDS,SUMMERFIELD URGENT CARE CENTRE,0,0,3571,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,MSitAE-MARCH-2024,RAN,NHS ENGLAND LONDON,ROYAL NATIONAL ORTHOPAEDIC HOSPITAL NHS TRUST,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,34
2,MSitAE-MARCH-2024,RC9,NHS ENGLAND EAST OF ENGLAND,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,16063,0,8681,0,0,699,...,0,0,0,0,82,6,4818,0,0,2322
3,MSitAE-MARCH-2024,AD913,NHS ENGLAND LONDON,BECKENHAM BEACON UCC,0,0,4049,0,0,192,...,37,0,0,0,0,0,0,0,0,0
4,MSitAE-MARCH-2024,NTV0B,NHS ENGLAND SOUTH EAST,ASHFORD WALK-IN-CENTRE,0,0,2651,0,0,0,...,75,0,0,0,0,0,0,0,0,0


In [2]:
# List every column name, numbered
for i, col in enumerate(df.columns):
    print(i, "->", col)

0 -> Period
1 -> Org Code
2 -> Parent Org
3 -> Org name
4 -> A&E attendances Type 1
5 -> A&E attendances Type 2
6 -> A&E attendances Other A&E Department
7 -> A&E attendances Booked Appointments Type 1
8 -> A&E attendances Booked Appointments Type 2
9 -> A&E attendances Booked Appointments Other Department
10 -> Attendances over 4hrs Type 1
11 -> Attendances over 4hrs Type 2
12 -> Attendances over 4hrs Other Department
13 -> Attendances over 4hrs Booked Appointments Type 1
14 -> Attendances over 4hrs Booked Appointments Type 2
15 -> Attendances over 4hrs Booked Appointments Other Department
16 -> Patients who have waited 4-12 hs from DTA to admission
17 -> Patients who have waited 12+ hrs from DTA to admission
18 -> Emergency admissions via A&E - Type 1
19 -> Emergency admissions via A&E - Type 2
20 -> Emergency admissions via A&E - Other A&E department
21 -> Other emergency admissions


In [3]:
# --- Stage 2: Clean and build the metric ---

# 1. Keep only the columns we need, and rename them to short, clean names
clean = df[[
    "Parent Org",
    "Org name",
    "A&E attendances Type 1",
    "A&E attendances Type 2",
    "A&E attendances Other A&E Department",
    "Attendances over 4hrs Type 1",
    "Attendances over 4hrs Type 2",
    "Attendances over 4hrs Other Department",
]].copy()

clean.columns = [
    "region", "trust",
    "att_type1", "att_type2", "att_other",
    "over4_type1", "over4_type2", "over4_other",
]

# 2. Total attendances and total over-4hr waits per trust
clean["total_attendances"] = clean["att_type1"] + clean["att_type2"] + clean["att_other"]
clean["total_over_4hrs"]  = clean["over4_type1"] + clean["over4_type2"] + clean["over4_other"]

# 3. Filter out tiny orgs (keep only real A&E providers with a meaningful number of attendances)
clean = clean[clean["total_attendances"] >= 100].copy()

# 4. Calculate % within 4 hours
clean["pct_within_4hrs"] = (
    (clean["total_attendances"] - clean["total_over_4hrs"]) / clean["total_attendances"] * 100
).round(1)

# 5. Add the month so we can combine files later
clean["month"] = "March 2024"

# Show the result: how many trusts remain, and a preview
print("Trusts remaining after cleaning:", clean.shape[0])
clean[["region", "trust", "total_attendances", "total_over_4hrs", "pct_within_4hrs"]].head(10)

Trusts remaining after cleaning: 183


,region,trust,total_attendances,total_over_4hrs,pct_within_4hrs
0,NHS ENGLAND MIDLANDS,SUMMERFIELD URGENT CARE CENTRE,3571,0,100.0
2,NHS ENGLAND EAST OF ENGLAND,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,24744,6131,75.2
3,NHS ENGLAND LONDON,BECKENHAM BEACON UCC,4049,37,99.1
4,NHS ENGLAND SOUTH EAST,ASHFORD WALK-IN-CENTRE,2651,75,97.2
5,NHS ENGLAND SOUTH EAST,WOKING WALK IN CENTRE,2771,180,93.5
7,NHS ENGLAND LONDON,HOMERTON HEALTHCARE NHS FOUNDATION TRUST,11073,1946,82.4
8,NHS ENGLAND SOUTH EAST,SOUTHERN HEALTH NHS FOUNDATION TRUST,3243,74,97.7
9,NHS ENGLAND NORTH EAST AND YORKSHIRE,CALDERDALE AND HUDDERSFIELD NHS FOUNDATION TRUST,15474,3637,76.5
10,NHS ENGLAND NORTH WEST,BLACKPOOL TEACHING HOSPITALS NHS FOUNDATION TRUST,20131,4281,78.7
11,NHS ENGLAND MIDLANDS,LINCOLNSHIRE COMMUNITY HEALTH SERVICES NHS TRUST,10692,780,92.7


In [4]:
# Refine: keep only real major A&E providers (meaningful Type 1 activity)
major = clean[clean["att_type1"] >= 100].copy()

# Recalculate the metric using TOTAL attendances (Type 1 is the bulk for these)
print("Major A&E trusts remaining:", major.shape[0])

# Sort to see the worst performers - the trusts missing the target most
major_sorted = major.sort_values("pct_within_4hrs")
major_sorted[["region", "trust", "total_attendances", "pct_within_4hrs"]].head(10)

Major A&E trusts remaining: 123


,region,trust,total_attendances,pct_within_4hrs
152,NHS ENGLAND MIDLANDS,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,13198,49.0
48,NHS ENGLAND NORTH WEST,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,7327,52.3
73,NHS ENGLAND NORTH WEST,EAST CHESHIRE NHS TRUST,4495,52.9
35,NHS ENGLAND SOUTH WEST,UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST,13638,53.9
150,NHS ENGLAND MIDLANDS,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,18026,54.2
141,NHS ENGLAND NORTH WEST,TAMESIDE AND GLOSSOP INTEGRATED CARE NHS FOUND...,11022,57.6
109,NHS ENGLAND SOUTH WEST,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,12817,57.9
114,NHS ENGLAND MIDLANDS,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,23594,58.4
178,NHS ENGLAND MIDLANDS,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,35395,59.5
125,NHS ENGLAND SOUTH WEST,DORSET COUNTY HOSPITAL NHS FOUNDATION TRUST,4473,59.9


In [5]:
# Regional view: average 4-hour performance by region
regional = major.groupby("region").agg(
    trusts=("trust", "count"),
    avg_pct_within_4hrs=("pct_within_4hrs", "mean"),
    total_attendances=("total_attendances", "sum"),
).round(1).sort_values("avg_pct_within_4hrs")

regional

,trusts,avg_pct_within_4hrs,total_attendances
region,,,
NHS ENGLAND NORTH WEST,18,66.6,297019
NHS ENGLAND SOUTH WEST,13,68.1,173477
NHS ENGLAND MIDLANDS,21,69.0,365311
NHS ENGLAND SOUTH EAST,17,71.8,281744
NHS ENGLAND NORTH EAST AND YORKSHIRE,22,72.8,338764
NHS ENGLAND EAST OF ENGLAND,13,73.6,231812
Total,1,73.8,2289703
NHS ENGLAND LONDON,18,74.3,379477


In [6]:
# Remove the NHS summary row - "Total" is not a real region
major = major[major["region"] != "Total"].copy()

# Rebuild the regional view without it, and tidy the region names
major["region"] = major["region"].str.replace("NHS ENGLAND ", "", regex=False).str.title()

regional = major.groupby("region").agg(
    trusts=("trust", "count"),
    avg_pct_within_4hrs=("pct_within_4hrs", "mean"),
).round(1).sort_values("avg_pct_within_4hrs")

regional

,trusts,avg_pct_within_4hrs
region,,
North West,18,66.6
South West,13,68.1
Midlands,21,69.0
South East,17,71.8
North East And Yorkshire,22,72.8
East Of England,13,73.6
London,18,74.3


In [7]:
# --- Turn our cleaning into a reusable function ---

def clean_ae_file(filename, month_label):
    df = pd.read_csv(filename)

    # The columns we need (same as before)
    needed = [
        "Parent Org", "Org name",
        "A&E attendances Type 1", "A&E attendances Type 2", "A&E attendances Other A&E Department",
        "Attendances over 4hrs Type 1", "Attendances over 4hrs Type 2", "Attendances over 4hrs Other Department",
    ]

    # Safety check: make sure every needed column exists
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"⚠️  {month_label}: MISSING COLUMNS -> {missing}")
        return None

    c = df[needed].copy()
    c.columns = ["region", "trust", "att1", "att2", "att_o", "o4_1", "o4_2", "o4_o"]

    c["total_attendances"] = c["att1"] + c["att2"] + c["att_o"]
    c["total_over_4hrs"]   = c["o4_1"] + c["o4_2"] + c["o4_o"]

    # Keep real major A&E providers only
    c = c[c["att1"] >= 100].copy()

    # Remove any NHS summary rows
    c = c[c["region"] != "Total"].copy()

    # The metric
    c["pct_within_4hrs"] = ((c["total_attendances"] - c["total_over_4hrs"]) / c["total_attendances"] * 100).round(1)

    # Tidy region names
    c["region"] = c["region"].str.replace("NHS ENGLAND ", "", regex=False).str.title()

    c["month"] = month_label
    print(f"✓ {month_label}: {c.shape[0]} major A&E trusts")
    return c

print("Function ready.")

Function ready.


In [8]:
# Run the cleaning function on all five March files
files = [
    ("March-2021-revised-280421-ce435.csv",   "March 2021"),
    ("March-2022-revised-120522-cab316.csv",  "March 2022"),
    ("March-2023-revised-110523-lll12c.csv",  "March 2023"),
    ("March-2024-revised-130624-abcdef123.csv","March 2024"),
    ("Monthly-AE-March-2025.csv",             "March 2025"),
]

# Clean each one and collect the results
all_years = []
for filename, label in files:
    result = clean_ae_file(filename, label)
    if result is not None:
        all_years.append(result)

# Stack them all into one combined table
combined = pd.concat(all_years, ignore_index=True)
print("\nCombined dataset:", combined.shape[0], "rows across", combined['month'].nunique(), "years")

✓ March 2021: 128 major A&E trusts
✓ March 2022: 126 major A&E trusts
✓ March 2023: 125 major A&E trusts
✓ March 2024: 122 major A&E trusts
✓ March 2025: 122 major A&E trusts

Combined dataset: 623 rows across 5 years


In [9]:
# National trend: average 4-hour performance per year
trend = combined.groupby("month").agg(
    trusts=("trust", "count"),
    avg_pct_within_4hrs=("pct_within_4hrs", "mean"),
).round(1)

trend

,trusts,avg_pct_within_4hrs
month,,
March 2021,128,86.6
March 2022,126,71.7
March 2023,125,70.9
March 2024,122,70.9
March 2025,122,71.4


In [10]:
# Export cleaned data for Power BI

# 1. The full trust-level data across all years (for detailed views)
combined.to_csv("ae_clean_all_years.csv", index=False)

# 2. The national trend by year
trend.to_csv("ae_trend_by_year.csv")

# 3. Regional performance by year (for the regional comparison)
regional_by_year = combined.groupby(["month", "region"]).agg(
    avg_pct_within_4hrs=("pct_within_4hrs", "mean"),
    trusts=("trust", "count"),
).round(1).reset_index()
regional_by_year.to_csv("ae_regional_by_year.csv", index=False)

print("✓ Three files exported:")
print("  - ae_clean_all_years.csv (every trust, every year)")
print("  - ae_trend_by_year.csv (national trend)")
print("  - ae_regional_by_year.csv (regional breakdown)")

✓ Three files exported:
  - ae_clean_all_years.csv (every trust, every year)
  - ae_trend_by_year.csv (national trend)
  - ae_regional_by_year.csv (regional breakdown)
